# Rehabilitacijski robot


Zagon:
   - IP: [10.20.0.23](https://10.20.0.23/desk/) 
   - Brake release
   - Activate FCI
   - Operations: Execution
   - Zagon ROS: docker restart rosmaster
   - Zagon kontrolerja: ssh pingvin@10.20.0.23 docker restart hri-vaje-controller


In [ ]:
#Inicializacija in uvozi
import rospy
ns = "pingvin_1"
rospy.init_node(ns)
from robotblockset.ros.franka import panda_ros
r = panda_ros(ns=ns, control_strategy="JointImpedance", init_node=False)
import inspect
import utils
from utils import SoftSetJointCompliance
import time
from robotblockset.transformations import rot_x, rp2t, q2r
from robotblockset.robots import isrobot
from robotblockset.graphics import plotjtraj, plotctraj
import matplotlib.pyplot as plt
import numpy as np
from importlib import reload
from dmp import DMP


In [ ]:
#Obnova po napaki
r.ErrorRecovery()

0

In [ ]:
# Ojačanja v kontrolerju nastavi na 0, robot bo kompenziral le gravitacijo
r.SetJointCompliant() 

:Joint compliance changed 
Stiff:[ 0.0000  0.0000  0.0000  0.0000  0.0000  0.0000  0.0000]
Damp:[ 0.0000  0.0000  0.0000  0.0000  0.0000  0.0000  0.0000]


In [ ]:
# Postopno višanje togosti robota.
r.ResetCurrentTarget()
SoftSetJointCompliance(r,r._franka_default.JointCompliance.K,4)

:Joint compliance changed 
Stiff:[ 1200.0000  1200.0000  1200.0000  1200.0000  0.0000  0.0000  0.0000]
Damp:[25 25 25 25 10 10 10]


In [ ]:
#Funkcija za snemanje  
def record_robot_state(robot, frequency, duration):
    interval = 1.0 / frequency
    N = int(frequency * duration)
    
    tt = np.zeros((N, 1))
    qt = np.zeros((N, robot.nj))
    dqt = np.zeros((N, robot.nj))

    i = 0
    start_time = time.monotonic()
    last_update = start_time

    while i < N:
        t = time.monotonic()
        if t - last_update >= interval:
            robot.GetState()
            
            tt[i] = t - start_time
            qt[i] = robot.q
            dqt[i] = robot.qdot

            last_update = t
            i += 1
        
    return tt, qt, dqt

In [ ]:
#Snemanje in shranjevanje trajektorije
time.sleep(5) 
print("Začetek snemanja stanja robota...")
trajektorija = 1
masa = 3
tt, qt, dqt = record_robot_state(r, 100, 10)
np.savez(
    f"trajektorija_{trajektorija}_masa_{masa}.npz",
    tt=tt,
    qt=qt,
    dqt=dqt,
    trajektorija=trajektorija,
    masa=masa
)
print(f"Shranjeno: trajektorija_{trajektorija}_masa_{masa}.npz")
print("Snemanje končano.")


In [ ]:
# Risanje trajektorije
ime_datoteke = "pot_1_teza_3.npz"

data = np.load(ime_datoteke)
tt = data["tt"]
qt = data["qt"]
dqt = data["dqt"]

fig, ax = plt.subplots(3, 1, figsize=(9, 6))
fig.suptitle(ime_datoteke)
plotjtraj(tt, qt, dqt, ax=ax)
plt.show()

In [ ]:
# Funkcija za nalaganje podatkov iz več datotek
def load_one_trajectory_all_masses(trajektorija=1, mase=range(1, 7)):
    podatki = {}

    for masa in mase:
        ime = f"trajektorija_{trajektorija}_masa_{masa}.npz"
        data = np.load(ime)

        podatki[masa] = {
            "tt": data["tt"].reshape(-1),
            "qt": data["qt"],
            "dqt": data["dqt"],
        }

    return podatki

In [ ]:
# Naložimo podatke za trajektorijo  in vse mase
podatki = load_one_trajectory_all_masses(trajektorija=1)

In [ ]:
# Funkcija za ponovni vzorčenje trajektorije na enakomernih časovnih intervalih
def resample_joint_traj(tt, qt, n_points=200):
    t_old = tt - tt[0]
    if t_old[-1] == 0:
        t_old[-1] = 1e-6

    t_new = np.linspace(0, t_old[-1], n_points)
    q_new = np.zeros((n_points, qt.shape[1]))

    for j in range(qt.shape[1]):
        q_new[:, j] = np.interp(t_new, t_old, qt[:, j])

    return t_new, q_new

In [ ]:
# Funkcija za pripravo povprečne trajektorije in standardnega odklona
def prepare_mean_trajectory(podatki, n_points=200):
    vse_q = []

    for masa in sorted(podatki.keys()):
        tt = podatki[masa]["tt"]
        qt = podatki[masa]["qt"]

        t_new, q_new = resample_joint_traj(tt, qt, n_points=n_points)
        vse_q.append(q_new)

    Q = np.stack(vse_q, axis=0)      # (st_mas, n_points, nj)
    q_mean = np.mean(Q, axis=0)
    q_std = np.std(Q, axis=0)
    dq_mean = np.gradient(q_mean, t_new, axis=0)

    return t_new, Q, q_mean, q_std, dq_mean

In [ ]:
# Priprava povprečne trajektorije in standardnega odklona
t_ref, Q, q_mean, q_std, dq_mean = prepare_mean_trajectory(podatki, n_points=200)

In [ ]:
# Funkcija za risanje vseh trajektorij in povprečne trajektorije
def plot_all_masses_and_mean(t_ref, Q, q_mean):
    n_joints = Q.shape[2]

    fig, ax = plt.subplots(n_joints, 1, figsize=(10, 2*n_joints), sharex=True)
    fig.suptitle("Vse mase + povprečna trajektorija")

    for j in range(n_joints):
        for k in range(Q.shape[0]):
            ax[j].plot(t_ref, Q[k, :, j], alpha=0.35)
        ax[j].plot(t_ref, q_mean[:, j], linewidth=3)
        ax[j].set_ylabel(f"q{j+1}")

    ax[-1].set_xlabel("čas [s]")
    plt.show()

In [ ]:
# Risanje vseh trajektorij in povprečne trajektorije
plot_all_masses_and_mean(t_ref, Q, q_mean)

In [ ]:
#Nauči DMP iz povprečne trajektorije
def train_dmp_from_mean(t_ref, q_mean, num_weights=25):
    dmp_model = DMP(pos_data=q_mean, time=t_ref, num_weights=num_weights)
    q_dmp, t_dmp = dmp_model.decode()
    t_dmp = np.array(t_dmp)
    return dmp_model, q_dmp, t_dmp

In [ ]:
# Nauči DMP iz povprečne trajektorije
dmp_model, q_dmp, t_dmp = train_dmp_from_mean(t_ref, q_mean, num_weights=25)

In [ ]:
#Primerjava povprečja in DMP
def plot_mean_vs_dmp(t_ref, q_mean, t_dmp, q_dmp):
    n_joints = q_mean.shape[1]

    fig, ax = plt.subplots(n_joints, 1, figsize=(10, 2*n_joints), sharex=False)
    fig.suptitle("Povprečna trajektorija in DMP aproksimacija")

    for j in range(n_joints):
        ax[j].plot(t_ref, q_mean[:, j], label="povprečje")
        ax[j].plot(t_dmp, q_dmp[:, j], "--", label="DMP")
        ax[j].set_ylabel(f"q{j+1}")
        ax[j].legend()

    ax[-1].set_xlabel("čas [s]")
    plt.show()

In [ ]:
#  Primerjava povprečja in DMP
plot_mean_vs_dmp(t_ref, q_mean, t_dmp, q_dmp)

In [ ]:
#Izračun napake DMP aproksimacije
def calculate_dmp_error(t_ref, q_mean, t_dmp, q_dmp):
    q_dmp_resampled = resample_q_only(t_dmp, q_dmp, t_ref)
    error = np.mean((q_mean - q_dmp_resampled) ** 2)
    return error

In [ ]:
#Izračun napake DMP aproksimacije
joint_rmse, total_rmse = calculate_dmp_error(t_ref, q_mean, t_dmp, q_dmp)

print("RMSE po sklepih:", joint_rmse)
print("Skupni RMSE:", total_rmse)